
# Simulação Wolf Sheep Simple 3

Este notebook implementa uma versão em Python do modelo **Wolf Sheep Simple 3**, inspirado em modelos de sistemas complexos e autômatos baseados em agentes.

O objetivo é simular a dinâmica entre:

- **Ovelhas**, que comem grama para ganhar energia.
- **Lobos**, que caçam ovelhas para ganhar energia.
- **Grama**, que cresce nos patches ao longo do tempo.
- **Energia**, que diminui com o movimento e aumenta com alimentação.
- **Reprodução**, que permite crescimento populacional.
- **Extinção**, quando uma espécie fica sem indivíduos.

Esse tipo de modelo é útil para estudar equilíbrio ecológico, competição por recursos, predação e dinâmicas não lineares.



## 1. Ideia geral do modelo

O modelo representa um ambiente bidimensional em grade. Cada célula da grade pode conter uma quantidade de grama. Ovelhas e lobos se movimentam aleatoriamente pelo espaço.

A cada passo da simulação:

1. As ovelhas se movem.
2. As ovelhas gastam energia.
3. As ovelhas comem grama, se houver grama suficiente.
4. As ovelhas podem se reproduzir.
5. Ovelhas sem energia morrem.
6. Os lobos se movem.
7. Os lobos gastam energia.
8. Os lobos comem ovelhas quando encontram alguma no mesmo patch.
9. Os lobos podem se reproduzir.
10. Lobos sem energia morrem.
11. A grama cresce novamente.
12. As populações são registradas.

A dinâmica emergente pode gerar ciclos populacionais: primeiro crescem as ovelhas, depois crescem os lobos, depois as ovelhas diminuem, e assim por diante.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import random

plt.rcParams["figure.figsize"] = (8, 6)



## 2. Parâmetros da simulação

Os parâmetros abaixo controlam o comportamento do sistema.

Você pode alterar os valores para observar diferentes cenários, por exemplo:

- Mais lobos no início.
- Menor crescimento da grama.
- Maior custo de movimento.
- Maior ou menor chance de reprodução.
- Mais energia obtida ao comer.


In [ ]:

# Tamanho do mundo
GRID_SIZE = 40

# Populações iniciais
INITIAL_SHEEP = 120
INITIAL_WOLVES = 40

# Energia inicial
INITIAL_SHEEP_ENERGY = 20
INITIAL_WOLF_ENERGY = 30

# Custo de movimento
SHEEP_MOVEMENT_COST = 1
WOLF_MOVEMENT_COST = 1

# Ganho de energia
ENERGY_GAIN_FROM_GRASS = 4
ENERGY_GAIN_FROM_SHEEP = 20

# Grama
MAX_GRASS = 10
GRASS_REGROWTH_RATE = 0.5

# Reprodução
SHEEP_REPRODUCTION_PROB = 0.04
WOLF_REPRODUCTION_PROB = 0.03

# Número de passos
N_STEPS = 300

# Semente para reprodutibilidade
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)



## 3. Classe dos agentes

Cada agente terá:

- `x` e `y`: posição na grade.
- `energy`: energia atual.
- `kind`: tipo do agente, podendo ser `"sheep"` ou `"wolf"`.

A movimentação é aleatória, usando uma vizinhança de Moore simplificada: o agente pode andar uma célula em qualquer direção.


In [ ]:

class Agent:
    def __init__(self, x, y, energy, kind):
        self.x = x
        self.y = y
        self.energy = energy
        self.kind = kind

    def move(self, grid_size):
        dx = random.choice([-1, 0, 1])
        dy = random.choice([-1, 0, 1])

        self.x = (self.x + dx) % grid_size
        self.y = (self.y + dy) % grid_size

    def position(self):
        return self.x, self.y



## 4. Função de inicialização

A função `setup()` cria:

- A matriz de grama.
- A lista inicial de ovelhas.
- A lista inicial de lobos.

A grama começa com valores aleatórios entre 0 e 10.


In [ ]:

def setup():
    grass = np.random.uniform(0, MAX_GRASS, size=(GRID_SIZE, GRID_SIZE))

    sheep = [
        Agent(
            x=random.randrange(GRID_SIZE),
            y=random.randrange(GRID_SIZE),
            energy=INITIAL_SHEEP_ENERGY,
            kind="sheep"
        )
        for _ in range(INITIAL_SHEEP)
    ]

    wolves = [
        Agent(
            x=random.randrange(GRID_SIZE),
            y=random.randrange(GRID_SIZE),
            energy=INITIAL_WOLF_ENERGY,
            kind="wolf"
        )
        for _ in range(INITIAL_WOLVES)
    ]

    return grass, sheep, wolves



## 5. Regras das ovelhas

As ovelhas:

1. Movem-se aleatoriamente.
2. Perdem energia pelo movimento.
3. Comem grama quando há grama suficiente.
4. Podem se reproduzir.
5. Morrem quando a energia fica menor ou igual a zero.


In [ ]:

def update_sheep(sheep, grass):
    new_sheep = []
    survivors = []

    for s in sheep:
        s.move(GRID_SIZE)
        s.energy -= SHEEP_MOVEMENT_COST

        x, y = s.position()

        if grass[x, y] >= ENERGY_GAIN_FROM_GRASS:
            s.energy += ENERGY_GAIN_FROM_GRASS
            grass[x, y] -= ENERGY_GAIN_FROM_GRASS

        if random.random() < SHEEP_REPRODUCTION_PROB and s.energy > 2:
            child_energy = s.energy / 2
            s.energy = s.energy / 2
            new_sheep.append(Agent(s.x, s.y, child_energy, "sheep"))

        if s.energy > 0:
            survivors.append(s)

    survivors.extend(new_sheep)
    return survivors



## 6. Regras dos lobos

Os lobos:

1. Movem-se aleatoriamente.
2. Perdem energia pelo movimento.
3. Procuram ovelhas na mesma posição.
4. Se encontrarem uma ovelha, comem essa ovelha e ganham energia.
5. Podem se reproduzir.
6. Morrem quando a energia fica menor ou igual a zero.


In [ ]:

def update_wolves(wolves, sheep):
    new_wolves = []
    wolf_survivors = []

    sheep_by_position = {}
    for i, s in enumerate(sheep):
        sheep_by_position.setdefault(s.position(), []).append(i)

    eaten_sheep_indices = set()

    for w in wolves:
        w.move(GRID_SIZE)
        w.energy -= WOLF_MOVEMENT_COST

        pos = w.position()

        if pos in sheep_by_position:
            available_sheep = [
                idx for idx in sheep_by_position[pos]
                if idx not in eaten_sheep_indices
            ]

            if available_sheep:
                chosen = available_sheep[0]
                eaten_sheep_indices.add(chosen)
                w.energy += ENERGY_GAIN_FROM_SHEEP

        if random.random() < WOLF_REPRODUCTION_PROB and w.energy > 2:
            child_energy = w.energy / 2
            w.energy = w.energy / 2
            new_wolves.append(Agent(w.x, w.y, child_energy, "wolf"))

        if w.energy > 0:
            wolf_survivors.append(w)

    remaining_sheep = [
        s for i, s in enumerate(sheep)
        if i not in eaten_sheep_indices
    ]

    wolf_survivors.extend(new_wolves)
    return wolf_survivors, remaining_sheep



## 7. Crescimento da grama

A grama cresce a cada passo até atingir o limite máximo definido por `MAX_GRASS`.


In [ ]:

def regrow_grass(grass):
    grass += GRASS_REGROWTH_RATE
    grass[grass > MAX_GRASS] = MAX_GRASS
    return grass



## 8. Visualização do ambiente

A função abaixo desenha:

- A grama em escala de fundo.
- As ovelhas como pontos claros.
- Os lobos como pontos escuros.


In [ ]:

def plot_world(grass, sheep, wolves, step):
    plt.figure(figsize=(7, 7))
    plt.imshow(grass.T, origin="lower", vmin=0, vmax=MAX_GRASS)

    if sheep:
        sheep_x = [s.x for s in sheep]
        sheep_y = [s.y for s in sheep]
        plt.scatter(sheep_x, sheep_y, s=12, label="Ovelhas", alpha=0.8)

    if wolves:
        wolf_x = [w.x for w in wolves]
        wolf_y = [w.y for w in wolves]
        plt.scatter(wolf_x, wolf_y, s=18, marker="x", label="Lobos", alpha=0.9)

    plt.title(f"Wolf Sheep Simple 3 - passo {step}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend(loc="upper right")
    plt.show()



## 9. Execução da simulação

A função `run_simulation()` executa o modelo e registra:

- Quantidade de ovelhas.
- Quantidade de lobos.
- Quantidade média de grama.
- Energia média das ovelhas.
- Energia média dos lobos.

Esses indicadores ajudam a analisar a dinâmica do sistema.


In [ ]:

def run_simulation(n_steps=N_STEPS, show_every=50):
    grass, sheep, wolves = setup()

    history = []

    for step in range(n_steps + 1):
        sheep_count = len(sheep)
        wolf_count = len(wolves)

        mean_grass = float(np.mean(grass))
        mean_sheep_energy = np.mean([s.energy for s in sheep]) if sheep else 0
        mean_wolf_energy = np.mean([w.energy for w in wolves]) if wolves else 0

        history.append({
            "step": step,
            "sheep": sheep_count,
            "wolves": wolf_count,
            "mean_grass": mean_grass,
            "mean_sheep_energy": mean_sheep_energy,
            "mean_wolf_energy": mean_wolf_energy
        })

        if step % show_every == 0:
            clear_output(wait=True)
            plot_world(grass, sheep, wolves, step)

        if sheep_count == 0 and wolf_count == 0:
            print(f"Todas as populações foram extintas no passo {step}.")
            break

        sheep = update_sheep(sheep, grass)
        wolves, sheep = update_wolves(wolves, sheep)
        grass = regrow_grass(grass)

    return pd.DataFrame(history), grass, sheep, wolves



## 10. Rodando o modelo


In [ ]:

df_history, final_grass, final_sheep, final_wolves = run_simulation(
    n_steps=N_STEPS,
    show_every=50
)

df_history.head()



## 11. Gráfico das populações

Este gráfico mostra a evolução temporal das populações.

Em muitos cenários, as populações oscilam:

- Quando há muita grama, as ovelhas aumentam.
- Quando há muitas ovelhas, os lobos têm mais alimento e aumentam.
- Quando os lobos aumentam muito, as ovelhas diminuem.
- Quando as ovelhas diminuem, os lobos ficam sem alimento e também diminuem.


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(df_history["step"], df_history["sheep"], label="Ovelhas")
plt.plot(df_history["step"], df_history["wolves"], label="Lobos")
plt.xlabel("Passo")
plt.ylabel("População")
plt.title("Dinâmica populacional: ovelhas e lobos")
plt.legend()
plt.grid(True)
plt.show()



## 12. Gráfico da grama média

Este gráfico mostra a quantidade média de grama no ambiente.

A grama é o recurso primário do sistema. Se ela ficar muito baixa, as ovelhas perdem energia e podem morrer. Como consequência, os lobos também podem ser afetados indiretamente.


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(df_history["step"], df_history["mean_grass"], label="Grama média")
plt.xlabel("Passo")
plt.ylabel("Quantidade média de grama")
plt.title("Evolução da grama média")
plt.legend()
plt.grid(True)
plt.show()



## 13. Energia média dos agentes

A energia é uma variável central do modelo.

Ela funciona como um estoque interno:

- O agente perde energia ao se mover.
- O agente ganha energia ao se alimentar.
- Se a energia acabar, o agente morre.
- Na reprodução, a energia é dividida entre o agente original e o novo agente.


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(df_history["step"], df_history["mean_sheep_energy"], label="Energia média das ovelhas")
plt.plot(df_history["step"], df_history["mean_wolf_energy"], label="Energia média dos lobos")
plt.xlabel("Passo")
plt.ylabel("Energia média")
plt.title("Energia média por tipo de agente")
plt.legend()
plt.grid(True)
plt.show()



## 14. Tabela final de resultados


In [ ]:

df_history.tail()



## 15. Interpretação

Este modelo mostra como regras simples podem gerar comportamentos complexos.

A dinâmica depende fortemente dos parâmetros. Pequenas mudanças no crescimento da grama, no custo de movimento ou na taxa de reprodução podem levar a resultados muito diferentes, como:

- Estabilidade populacional.
- Oscilações entre predador e presa.
- Explosão populacional das ovelhas.
- Extinção dos lobos.
- Extinção das ovelhas.
- Colapso completo do sistema.

Esse comportamento é característico de sistemas complexos: o resultado global não é simplesmente a soma das regras locais, mas emerge da interação entre agentes e ambiente.



## 16. Experimentos sugeridos

Altere os parâmetros e observe os resultados:

1. Aumente `INITIAL_WOLVES`.
2. Reduza `GRASS_REGROWTH_RATE`.
3. Aumente `SHEEP_REPRODUCTION_PROB`.
4. Aumente `ENERGY_GAIN_FROM_SHEEP`.
5. Reduza `ENERGY_GAIN_FROM_GRASS`.
6. Aumente `WOLF_MOVEMENT_COST`.

Depois compare os gráficos de população, grama e energia.
